# 02 — Carregar as bases públicas (CNEFE, CNAE e CEPs dos Correios)

**Objetivo:** ler o CNEFE do município, a tabela de CNAE (subclasses) e a base de CEPs dos Correios (só a cidade).

**Entradas:** `01_dados_brutos/cnefe/`, `01_dados_brutos/cnae/`, `01_dados_brutos/correios/TB_CEP_BR_2018.csv`.

**Saídas:** `03_processamento/02_carregar_bases_publicas/`: `df_cnefe`, `df_cnae`, `df_base_cep` (Parquet + manifesto).

**Como rodar:** de cima para baixo. Usa o cache quando ele vale.

**Seções:** 1. CNEFE · 2. Tabela de CNAE · 3. Base de CEPs dos Correios · 4. Validação · 5. Resumo

In [ ]:
# ===================================================================
# PREPARAÇÃO: bibliotecas, config.yaml, log e entradas
# ===================================================================
# Bibliotecas que já vêm com o Python.
import datetime          # datas e horas
import importlib.util    # conferir se uma biblioteca está instalada, sem carregá-la
import json              # ler e gravar arquivos JSON
import os                # trocar arquivos de lugar com segurança
import re                # achar trechos de texto por padrão
import sys               # deixar o Python enxergar a pasta scripts/src
import time              # medir quanto tempo cada etapa leva
from pathlib import Path  # caminhos de pastas e arquivos

# Bibliotecas de cálculos e de tabelas (DataFrames).
import numpy as np
import pandas as pd

# Acha a raiz do projeto (a pasta do config.yaml), subindo a partir da pasta atual,
# e deixa o Python enxergar as funções comuns do projeto (scripts/src).
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists())
if str(RAIZ / "scripts") not in sys.path:
    sys.path.insert(0, str(RAIZ / "scripts"))

# Funções comuns: config.yaml, log, validação, arquivos de outras etapas.
from src import config, fontes, registro, validacao
from src.registro import formatar_tempo

# Lê o config.yaml e inicia o log (na tela e em 03_processamento/logs).
cfg = config.carregar(RAIZ)
log = registro.iniciar_log("02_carregar_bases_publicas", cfg)
# Hora de início do notebook (para o tempo total no resumo final).
inicio_notebook = time.time()
log.info("Preparando o notebook...")

# ===== PARÂMETROS (config.yaml) =====
COD_MUNICIPIO_IBGE = cfg.projeto("municipio_ibge7")          # código IBGE (início do nome do arquivo do CNEFE)
CHUNK = cfg.parametro("leitura.chunk")                       # linhas lidas por vez
# CNEFE
PASTA_CNEFE = cfg.caminho("brutos.cnefe")
PADRAO_CNEFE = f"{COD_MUNICIPIO_IBGE}*"                      # arquivos que começam com o código do IBGE
SEP_CNEFE = cfg.parametro("cnefe.separador")
ENC_CNEFE = cfg.parametro("cnefe.codificacao")
# CNAE
PASTA_CNAE = cfg.caminho("brutos.cnae")
PADRAO_CNAE = cfg.parametro("cnae.padrao")
ABA_CNAE = cfg.parametro("cnae.aba")
# Base de CEPs dos Correios
ARQ_CEP = cfg.caminho("brutos.correios")
SEP_CEP = cfg.parametro("correios.separador")
ENC_CEP = cfg.parametro("correios.codificacao")
UF_BASE_CEP = cfg.parametro("correios.uf")
CIDADE_BASE_CEP = cfg.parametro("correios.cidade")

# ===== CACHE (Parquet + manifesto, na pasta 03_processamento/02_carregar_bases_publicas) =====
from src.cache import CacheTabelas, impressao_arquivos


def descrever_cache(nome):
    """Devolve (origem, parâmetros, dependências) de cada tabela: é isso que decide se o cache vale."""
    if nome == "df_cnefe":
        return (impressao_arquivos(PASTA_CNEFE, PADRAO_CNEFE),
                {"separador": SEP_CNEFE, "codificacao": ENC_CNEFE, "cod_municipio_ibge": COD_MUNICIPIO_IBGE}, None)
    if nome == "df_cnae":
        return impressao_arquivos(PASTA_CNAE, PADRAO_CNAE), {"aba": ABA_CNAE}, None
    if nome == "df_base_cep":
        return (impressao_arquivos(ARQ_CEP.parent, ARQ_CEP.name),
                {"separador": SEP_CEP, "codificacao": ENC_CEP, "uf": UF_BASE_CEP, "cidade": CIDADE_BASE_CEP}, None)
    raise KeyError(f"Tabela sem descrição de cache: {nome}")


cache = CacheTabelas(
    pasta=cfg.pasta_etapa("02_carregar_bases_publicas"),
    versoes=cfg.parametro("cache.versao_logica"),
    descrever=descrever_cache,
    forcar=cfg.parametro("cache.forcar_reprocessar", []),
    idade_maxima_dias=cfg.parametro("cache.idade_maxima_dias", {}),
)
# Mesmos nomes de antes, para as seções continuarem iguais.
verificar_cache, carregar_cache, salvar_cache = cache.verificar, cache.carregar, cache.salvar

# ===== ENTRADAS =====
log.info(f"CNEFE: {len(impressao_arquivos(PASTA_CNEFE, PADRAO_CNEFE))} arquivo(s) em {PASTA_CNEFE}")
log.info(f"CNAE: {len(impressao_arquivos(PASTA_CNAE, PADRAO_CNAE))} arquivo(s) em {PASTA_CNAE}")
log.info(f"CEPs dos Correios: {ARQ_CEP} ({'existe' if ARQ_CEP.is_file() else 'NÃO existe'})")

log.info(f"Preparação concluída! ({formatar_tempo(time.time() - inicio_notebook)})")

## 1. CNEFE (IBGE)
Lê o arquivo do CNEFE do município e guarda tudo em `df_cnefe`. Se o cache vale, carrega dele.

In [ ]:
log.info("SEÇÃO 1: CNEFE (usa o cache quando ele vale)...")
inicio_b5 = time.time()

# Confere se o cache da tabela vale (arquivos de origem, parâmetros, versão e dependências iguais aos de quando foi gravado).
log.info("Conferindo o cache de df_cnefe...")
usar_cache, motivo_cache = verificar_cache("df_cnefe")
log.info("Conferência do cache concluída!")

if usar_cache:
    log.info(f"Usando o cache ({motivo_cache}): carregando df_cnefe...")
    df_cnefe = carregar_cache("df_cnefe")
    log.info(f"df_cnefe carregado do cache! {len(df_cnefe):,} linhas | {df_cnefe.shape[1]} colunas.")
else:
    log.info(f"Cache não usado ({motivo_cache}): lendo os arquivos originais...")
    log.info("Lendo CNEFE...")
    inicio_etapa = time.time()

    # Procura na pasta do IBGE os arquivos cujo nome começa com o código do município.
    arquivos = sorted(p for p in Path(PASTA_CNEFE).glob(PADRAO_CNEFE) if p.is_file())
    log.info(f"{len(arquivos)} arquivo(s) do CNEFE encontrado(s)!")

    partes = []

    for numero, arquivo in enumerate(arquivos, start=1):
        log.info(f"[{numero}/{len(arquivos)}] Lendo {arquivo.name}...")
        inicio_arquivo = time.time()
        total_arquivo = 0

        # Leitura em blocos, tudo como texto (dtype=str), como na inspeção.
        # Aqui NÃO há filtro: todas as linhas serão guardadas.
        leitor = pd.read_csv(
            arquivo,
            sep=SEP_CNEFE,
            encoding=ENC_CNEFE,
            dtype=str,
            chunksize=CHUNK,
        )

        for bloco in leitor:
            total_arquivo += len(bloco)
            # Guarda o bloco inteiro na lista.
            partes.append(bloco)
            log.info(f"    {total_arquivo:,} linhas lidas...")

        log.info(f"[{numero}/{len(arquivos)}] {arquivo.name} concluído! "
              f"{total_arquivo:,} linhas lidas "
              f"({formatar_tempo(time.time() - inicio_arquivo)}).")

    log.info("Juntando os pedaços em um único DataFrame (df_cnefe)...")
    if partes:
        df_cnefe = pd.concat(partes, ignore_index=True)
    else:
        df_cnefe = pd.DataFrame()

    log.info(f"Leitura concluída! df_cnefe: {len(df_cnefe):,} linhas | "
          f"{df_cnefe.shape[1]} colunas | tempo total: {formatar_tempo(time.time() - inicio_etapa)}.")

    # Grava o cache para as próximas sessões.
    salvar_cache("df_cnefe", df_cnefe)

log.info(f"\nSEÇÃO 1 concluída! Tempo total: {formatar_tempo(time.time() - inicio_b5)}.")

## 2. Tabela de CNAE (IBGE)
Lê a planilha da estrutura detalhada da CNAE-Subclasses (aba definida em `parametros.cnae.aba`, no config.yaml) e monta `df_cnae`, com uma linha por subclasse.
Cada subclasse fica com o código de 7 dígitos (usado para cruzar com a Receita), a denominação e, dos níveis acima, a seção e a divisão. Se o cache vale, carrega dele. Não exibe linhas do DataFrame, só logs.

In [ ]:
log.info("SEÇÃO 2: tabela de CNAE (usa o cache quando ele vale)...")
inicio_b8 = time.time()

# Confere se o cache da tabela vale (arquivos de origem, parâmetros, versão e dependências iguais aos de quando foi gravado).
log.info("Conferindo o cache de df_cnae...")
usar_cache, motivo_cache = verificar_cache("df_cnae")
log.info("Conferência do cache concluída!")

if usar_cache:
    log.info(f"Usando o cache ({motivo_cache}): carregando df_cnae...")
    df_cnae = carregar_cache("df_cnae")
    log.info(f"df_cnae carregado do cache! {len(df_cnae):,} linhas | {df_cnae.shape[1]} colunas.")
else:
    log.info(f"Cache não usado ({motivo_cache}): lendo os arquivos originais...")
    log.info("SEÇÃO 2: lendo a tabela de CNAE...")
    # Guarda a hora de início para calcular o tempo total no final.
    inicio_b8 = time.time()

    # O nome da aba com a estrutura detalhada (ABA_CNAE) fica no config.yaml.

    # ===================================================================
    # ETAPA 1: PROCURAR E LER A PLANILHA
    # ===================================================================
    log.info("Etapa 1: procurando e lendo a planilha...")

    # Procura na pasta do IBGE os arquivos que combinam com o padrão, em ordem alfabética.
    arquivos_cnae = sorted(p for p in Path(PASTA_CNAE).glob(PADRAO_CNAE) if p.is_file())
    if not arquivos_cnae:
        raise FileNotFoundError(f"Bloco interrompido: nenhuma planilha de CNAE encontrada em {PASTA_CNAE} "
                                f"com o padrão {PADRAO_CNAE}. Nada foi lido.")
    arquivo_cnae = arquivos_cnae[0]
    log.info(f"    Usando o arquivo: {arquivo_cnae.name}")

    # header=None: a planilha não tem uma linha de títulos aproveitável (o cabeçalho ocupa várias linhas).
    # dtype=str: lê tudo como texto, sem perder zeros à esquerda dos códigos.
    try:
        bruto = pd.read_excel(arquivo_cnae, sheet_name=ABA_CNAE, header=None, dtype=str)
    except ImportError:
        raise ImportError("Bloco interrompido: falta a biblioteca openpyxl para ler .xlsx "
                          "(py -m pip install openpyxl). Nada foi lido.")

    # As 6 colunas, na ordem em que aparecem na planilha.
    bruto.columns = ["secao", "divisao", "grupo", "classe", "subclasse", "denominacao"]

    log.info(f"Etapa 1 concluída! {len(bruto):,} linhas lidas da aba '{ABA_CNAE}'.")

    # ===================================================================
    # ETAPA 2: MONTAR A HIERARQUIA
    # ===================================================================
    log.info("\nEtapa 2: montando a hierarquia (seção, divisão e subclasse)...")

    # Cada linha da planilha preenche só o nível a que pertence; os demais ficam vazios.
    # Vamos percorrer as linhas de cima para baixo, guardando o último nível visto.
    secao_atual = ("", "")      # (código, denominação) da seção mais recente
    divisao_atual = ("", "")    # (código, denominação) da divisão mais recente
    registros = []              # lista onde montamos uma linha por subclasse

    # itertuples percorre a planilha linha a linha (é mais rápido que acessar célula a célula).
    for linha in bruto.itertuples(index=False):
        # A denominação é o texto do nível; sem ela, a linha é de cabeçalho ou está em branco.
        # Célula vazia chega como NaN (um número, e não texto), então só usamos o valor se for texto.
        denominacao = linha.denominacao.strip() if isinstance(linha.denominacao, str) else ""
        if not denominacao:
            continue
        # Linha de seção: uma única letra, de A a U (isso descarta a linha de título "Códigos").
        if isinstance(linha.secao, str) and len(linha.secao.strip()) == 1:
            secao_atual = (linha.secao.strip(), denominacao)
            continue
        # Linha de divisão: dois dígitos.
        if isinstance(linha.divisao, str) and linha.divisao.strip():
            divisao_atual = (linha.divisao.strip(), denominacao)
            continue
        # Linhas de grupo e de classe não são guardadas (você pediu só subclasse, seção e divisão).
        if isinstance(linha.grupo, str) and linha.grupo.strip():
            continue
        if isinstance(linha.classe, str) and linha.classe.strip():
            continue
        # Linha de subclasse: o código vem formatado, como "0111-3/01".
        if isinstance(linha.subclasse, str) and linha.subclasse.strip():
            formatado = linha.subclasse.strip()
            # Só os dígitos: é assim que o código aparece na base da Receita ("0111301").
            so_digitos = "".join(c for c in formatado if c.isdigit())
            # A linha do cabeçalho ("Subclasse") não tem 7 dígitos e é descartada aqui.
            if len(so_digitos) != 7:
                continue
            registros.append({
                "cnae_principal": so_digitos,
                "cnae_subclasse": formatado,
                "cnae_denominacao": denominacao,
                "cnae_secao": secao_atual[0],
                "cnae_secao_denominacao": secao_atual[1],
                "cnae_divisao": divisao_atual[0],
                "cnae_divisao_denominacao": divisao_atual[1],
            })

    # Transforma a lista de linhas em uma tabela.
    df_cnae = pd.DataFrame(registros)

    log.info(f"Etapa 2 concluída! {len(df_cnae):,} subclasses montadas.")

    # ===================================================================
    # ETAPA 3: CONFERÊNCIAS E RESUMO
    # ===================================================================
    log.info("\nEtapa 3: conferindo o resultado...")

    log.info("\n--- CONFERÊNCIAS ---")
    log.info(f"Códigos de 7 dígitos todos diferentes: "
          f"{'confere' if df_cnae['cnae_principal'].is_unique else 'NÃO confere'}")
    log.info(f"Toda subclasse tem seção e divisão: "
          f"{'confere' if ((df_cnae['cnae_secao'] != '') & (df_cnae['cnae_divisao'] != '')).all() else 'NÃO confere'}")

    log.info("\n--- RESUMO ---")
    log.info(f"df_cnae: {len(df_cnae):,} linhas | colunas: {list(df_cnae.columns)}")
    log.info(f"Seções distintas: {df_cnae['cnae_secao'].nunique():,} | divisões distintas: {df_cnae['cnae_divisao'].nunique():,}")

    log.info(f"\nSEÇÃO 2 concluída! Tempo total: {formatar_tempo(time.time() - inicio_b8)}.")

    # Grava o cache para as próximas sessões.
    salvar_cache("df_cnae", df_cnae)

log.info(f"\nSEÇÃO 2 concluída! Tempo total: {formatar_tempo(time.time() - inicio_b8)}.")

## 3. Base de CEPs dos Correios
Lê a base de CEPs (`brutos.correios`, de 2018) em partes e fica só com as linhas da cidade (`parametros.correios.uf` e `cidade`), guardando CEP, bairro e logradouro em `df_base_cep`. Se o cache vale, carrega dele.
Ela é usada a partir do notebook 05 (seção 1): para saber o bairro de cada CEP (desempatar vias com o mesmo nome), para achar o CEP de vias que só os Correios conhecem (passo 11) e para descobrir os CEPs não confiáveis.
Não exibe linhas do DataFrame, só logs.

In [ ]:
log.info("SEÇÃO 3: base de CEPs dos Correios (usa o cache quando ele vale)...")
inicio_b9 = time.time()

# Confere se o cache da tabela vale (arquivo de origem, parâmetros e versão iguais aos de quando foi gravado).
log.info("Conferindo o cache de df_base_cep...")
usar_cache, motivo_cache = verificar_cache("df_base_cep")
log.info("Conferência do cache concluída!")

if usar_cache:
    log.info(f"Usando o cache ({motivo_cache}): carregando df_base_cep...")
    df_base_cep = carregar_cache("df_base_cep")
    log.info(f"df_base_cep carregado do cache! {len(df_base_cep):,} linhas | {df_base_cep.shape[1]} colunas.")
else:
    log.info(f"Cache não usado ({motivo_cache}): lendo o arquivo original...")
    # Confere se o arquivo existe.
    caminho_cep = Path(ARQ_CEP)
    if not caminho_cep.is_file():
        raise FileNotFoundError(f"Bloco interrompido: não encontrei a base de CEPs em '{ARQ_CEP}'. "
                                "Ajuste ARQ_CEP no config.yaml. Nada foi lido.")

    # ===================================================================
    # ETAPA 1: LER O ARQUIVO E FICAR SÓ COM A CIDADE
    # ===================================================================
    log.info("Etapa 1: lendo a base de CEPs e ficando só com a cidade...")
    # Lista onde guardamos o pedaço da cidade de cada parte lida.
    partes_cep = []
    # Contador de linhas lidas, para o log.
    total_lido_cep = 0
    # Lê o arquivo em partes (ele é grande e traz o Brasil todo), tudo como texto.
    leitor_cep = pd.read_csv(caminho_cep, sep=SEP_CEP, encoding=ENC_CEP, dtype=str,
                             usecols=["CEP", "UF", "CIDADE", "BAIRRO", "LOGRADOURO"], chunksize=CHUNK)
    for pedaco in leitor_cep:
        total_lido_cep += len(pedaco)
        # Fica só com as linhas do estado e da cidade pedidos.
        da_cidade = (pedaco["UF"].str.strip() == UF_BASE_CEP) & (pedaco["CIDADE"].str.strip() == CIDADE_BASE_CEP)
        partes_cep.append(pedaco.loc[da_cidade, ["CEP", "BAIRRO", "LOGRADOURO"]])
        log.info(f"    {total_lido_cep:,} linhas lidas...")

    # Junta as partes numa tabela só e tira as linhas repetidas (a base repete algumas ruas).
    df_base_cep = pd.concat(partes_cep, ignore_index=True)
    del partes_cep
    if df_base_cep.empty:
        raise ValueError(f"Bloco interrompido: nenhuma linha de '{CIDADE_BASE_CEP}' ({UF_BASE_CEP}) na base de CEPs. "
                         "Confira UF_BASE_CEP e CIDADE_BASE_CEP no config.yaml.")
    linhas_antes = len(df_base_cep)
    df_base_cep = df_base_cep.drop_duplicates().reset_index(drop=True)
    log.info(f"Etapa 1 concluída! Linhas da cidade: {linhas_antes:,} | depois de tirar as repetidas: {len(df_base_cep):,}.")

    # Grava o cache para as próximas sessões.
    salvar_cache("df_base_cep", df_base_cep)

log.info("\n--- RESUMO ---")
log.info(f"df_base_cep: {len(df_base_cep):,} linhas | colunas: {list(df_base_cep.columns)}")
log.info(f"CEPs distintos da cidade: {df_base_cep['CEP'].nunique():,}")

log.info(f"\nSEÇÃO 3 concluída! Tempo total: {formatar_tempo(time.time() - inicio_b9)}.")

## 4. Validação
Linhas, nulos e duplicados de cada tabela. No CNEFE, as repetições do `COD_UNICO_ENDERECO` são diagnosticadas à parte: o log diz quantas têm a mesma coordenada (inofensivas) e quantas têm coordenadas diferentes (só estas geram aviso).


In [ ]:
log.info("SEÇÃO 4: validação das tabelas...")
validacao.validar_tabela("df_cnefe", df_cnefe, obrigatorias=["LATITUDE", "LONGITUDE"])
# O CNEFE repete alguns COD_UNICO_ENDERECO (o código já vem repetido do IBGE). O diagnóstico separa as repetições
# com a mesma coordenada (inofensivas) das que têm coordenadas diferentes (só estas merecem conferência).
validacao.diagnosticar_repeticoes("df_cnefe", df_cnefe, chave="COD_UNICO_ENDERECO", colunas_coordenadas=("LATITUDE", "LONGITUDE"))
validacao.validar_tabela("df_cnae", df_cnae, chave="cnae_principal", obrigatorias=["cnae_principal", "cnae_denominacao"])
validacao.validar_tabela("df_base_cep", df_base_cep, obrigatorias=["CEP", "LOGRADOURO"])
log.info("SEÇÃO 4 concluída!")

## 5. Resumo
Números principais, arquivos gerados e tempo total.

In [ ]:
log.info("RESUMO do 02_carregar_bases_publicas...")
log.info(f"CNEFE: {len(df_cnefe):,} endereços | CNAE: {len(df_cnae):,} subclasses | CEPs: {len(df_base_cep):,}")
log.info("Arquivos gerados:")
for nome_tabela in ['df_cnefe', 'df_cnae', 'df_base_cep']:
    caminho = cfg.caminho("processamento." + nome_tabela)
    log.info(f"    {caminho.relative_to(RAIZ)} ({caminho.stat().st_size / 1e6:.1f} MB)")
log.info(f"RESUMO concluído! Tempo total do notebook: {formatar_tempo(time.time() - inicio_notebook)}.")